In [32]:
import pandas as pd
import re
import spacy
import kagglehub

# MODEL_NAME = "Pulk17/Fake-News-Detection"
# FAKE_CLASS = 0
# TRUE_CLASS = 1

MODEL_NAME = "dhruvpal/fake-news-bert" 
FAKE_CLASS = 1
TRUE_CLASS = 0

nlp = spacy.load("en_core_web_sm")

def first_n_words(text, n=300):
    words = text.split()
    return ' '.join(words[:n])

def cut_to_n_sentences(text):
    doc = nlp(text)
    return ' '.join([sent.text for sent in (list(doc.sents)[:2] + list(doc.sents)[-2:])])

_CLEAN_RE = re.compile(r"[^A-Za-z\s]+")   # non-letters

def parse_text(df):
    return df.str.replace(_CLEAN_RE, "", regex=True)

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
real = pd.read_csv(f"{path}/True.csv")
fake = pd.read_csv(f"{path}/Fake.csv")

fake["label"] = FAKE_CLASS   # Fake
real["label"] = TRUE_CLASS   # Real

df_concat = pd.concat([fake, real], ignore_index=True)
df_concat["text"] = df_concat["title"] + " " + df_concat["text"]
df_concat["text"] = df_concat["text"].map(first_n_words)
df_concat = df_concat[["text", "label"]]

df = df_concat.sample(frac=1, random_state=42).reset_index(drop=True)

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.


In [33]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [34]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [35]:
import numpy as np

def _to_list_of_str(texts):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    if isinstance(texts, str):
        return [texts]

    return ["" if t is None else str(t) for t in texts]

def tokenize(texts):
    texts = _to_list_of_str(texts)
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

def predict(texts):
    texts = _to_list_of_str(texts)
    with torch.no_grad():
        inputs = tokenize(texts)
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return probs.cpu().numpy()

@torch.inference_mode()
def predict_torch(texts):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=256,
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    out = model(**enc)

    logits = out.logits if hasattr(out, "logits") else out[0]
    probs = torch.softmax(logits, dim=-1)

    return probs.detach().cpu().numpy()

def predict_tokens(tokens):
    with torch.no_grad():
        outputs = model(**tokens)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    return probs

In [36]:
import torch
import numpy as np
from tqdm import tqdm

def batch_encodings(test_encodings, first, last):
    batch_input_ids = test_encodings['input_ids'][first:last]
    batch_attention_mask = test_encodings['attention_mask'][first:last]
    # batch_token_type_ids = test_encodings['token_type_ids'][first:last]

    return {
        'input_ids': batch_input_ids,
        'attention_mask': batch_attention_mask,
        # 'token_type_ids': batch_token_type_ids
    }

TEST_N_BATCHES = 40

test_encodings = tokenize(test_texts)
test_encodings = {k: v.to(device) for k, v in test_encodings.items()}
batch_size = 128
test_labels = test_labels[:TEST_N_BATCHES * batch_size]

# Define a batch size for inference to prevent OutOfMemory errors
predictions = []

# Get the total number of test samples
num_samples = test_encodings['input_ids'].shape[0]

with torch.no_grad():
    # for i in tqdm(range(0, num_samples, batch_size)):
    for i in tqdm(range(0, batch_size * TEST_N_BATCHES, batch_size)):
        # Extract batch from test_encodings
        batch = batch_encodings(test_encodings, i, i + batch_size)

        # Perform inference on the batch
        outputs = model(**batch)
        batch_predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        predictions.extend(batch_predictions)

# Convert the list of predictions to a numpy array
predictions = np.array(predictions)

100%|██████████| 40/40 [00:06<00:00,  5.91it/s]


In [37]:
present_tokens = test_encodings['input_ids'] > 0
present_tokens = torch.sum(present_tokens, dim = -1, dtype = int)
# present_tokens = torch.tensor(sorted(present_tokens.tolist()))

print(torch.max(present_tokens))
print(torch.median(present_tokens))
print(torch.min(present_tokens))

tensor(256, device='cuda:0')
tensor(256, device='cuda:0')
tensor(14, device='cuda:0')


In [38]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(test_labels, predictions)
print(f"Test Accuracy: {accuracy:.4f}\n")
print(classification_report(test_labels, predictions, target_names=["Fake", "Real"]))

Test Accuracy: 0.9992

              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      2415
        Real       1.00      1.00      1.00      2705

    accuracy                           1.00      5120
   macro avg       1.00      1.00      1.00      5120
weighted avg       1.00      1.00      1.00      5120



In [39]:
import shap

masker = shap.maskers.Text(
    tokenizer=tokenizer,
    mask_token=tokenizer.mask_token,
)

explainer = shap.Explainer(
    predict_torch,
    masker,
    output_names=['fake', 'real'],
    algorithm='partition',
)

In [40]:
def top_words(shap_values, n=1, sample_idx=0, class_idx=0, top_k=10):
    dfs = []

    for sample_idx in range(n):
        values = shap_values.values[sample_idx][:, class_idx]
        tokens = shap_values.data[sample_idx]

        df = pd.DataFrame({
            "token": tokens,
            "importance": values
        })

        df = df[df.token.str.strip() != ""]
        df["abs"] = df.importance.abs()

        dfs.append(df.sort_values("abs", ascending=False).head(top_k))

    return dfs

In [41]:
import matplotlib.pyplot as plt

def plot_top_words(dfs):
    for i in range(len(dfs)):
        plt.figure(figsize=(8, 4))
        plt.barh(
            dfs[i].token[::-1],
            dfs[i].importance[::-1]
        )
        plt.axvline(0, color="black", linewidth=0.8)
        plt.title("Most Important Words (SHAP)")
        plt.xlabel("Impact on prediction")
        plt.tight_layout()
        plt.show()

# plot_top_words(top_words(shap_values, 2))

In [ ]:
import numpy as np
import copy

N_TEXTS = 50
ROUNDS = 5          
MAX_STEPS = 50
TARGETED = True
TARGET_CLASS = FAKE_CLASS
EPS = 1e-3

texts = None
if TARGET_CLASS == TRUE_CLASS:
    texts = df_concat.iloc[:int(2e4)].sample(frac=N_TEXTS/2e4, random_state=42).reset_index(drop=True)
    texts = texts["text"].dropna().tolist()
else:
    texts = df_concat.iloc[-int(2e4):].sample(frac=N_TEXTS/2e4, random_state=42).reset_index(drop=True)
    texts = texts["text"].dropna().tolist()

current_texts = copy.deepcopy(texts)
current_pred  = predict_torch(current_texts)
first_pred    = current_pred.copy()

def join_tokens(tokens):
    return "".join(tokens)

def current_target_class(i):
    return TARGET_CLASS if TARGETED else int(np.argmax(current_pred[i]))

def pick_ranking(shap_vals_i, class_idx, mode="pos"):
    v = shap_vals_i[:, class_idx]

    if mode == "abs":
        return np.argsort(-np.abs(v))
    else:
        return np.argsort(-v)

def compute_change(i, new_pred_i):
    if TARGETED:
        cls = TARGET_CLASS
        return new_pred_i[cls] - current_pred[i, cls]
    else:
        cls = int(np.argmax(current_pred[i]))
        return current_pred[i, cls] - new_pred_i[cls]

def check_flipped(new_pred_i):
    if TARGETED:
        cls = TARGET_CLASS
        return new_pred_i[cls] > new_pred_i[1-cls]

    else:
        return False

flipped = np.zeros(N_TEXTS, dtype=bool)

for round_num in range(ROUNDS):
    shap_values = explainer(current_texts)

    ranks = []
    toks_cache = []
    for i in range(N_TEXTS):
        cls = FAKE_CLASS 
        ranks.append(pick_ranking(shap_values.values[i], cls, mode="pos"))
        toks_cache.append(list(shap_values.data[i]))

    altered = np.zeros(N_TEXTS, dtype=bool)
    for step in range(MAX_STEPS):
        new_texts = current_texts.copy()

        for i in range(N_TEXTS):
            if flipped[i]:
                continue

            r = ranks[i]
            deleted_idx = int(r[step]) if step < len(r) else r[0]

            toks = toks_cache[i].copy()
            toks[deleted_idx] = ""
            new_texts[i] = join_tokens(toks)

        new_pred = predict_torch(new_texts)

        for i in range(N_TEXTS):
            if flipped[i]:
                continue

            flipped[i] = check_flipped(new_pred[i])
            if flipped[i]:
                print(f"Sample {i} flipped.")

            change = compute_change(i, new_pred[i])
            r = ranks[i]
            deleted_idx = int(r[step]) if step < len(r) else r[0]
            deleted_word = toks_cache[i][deleted_idx]

            if change > EPS:
                print(f"Deleting word {deleted_word} in sample {i} with attribution {shap_values.values[i][deleted_idx]} prediction change {change}")

                toks_cache[i][deleted_idx] = ""
                current_texts[i] = new_texts[i]
                current_pred[i]  = new_pred[i]
                altered[i] = True

    print("Altering unaltered samples")
    for i in range(N_TEXTS):
        if not altered[i] and not flipped[i]:
            deleted_idx = ranks[i][0]
            deleted_word = toks_cache[i][deleted_idx]

            # print(f"Deleting word {deleted_word} in sample {i} with attribution {shap_values.values[i][deleted_idx]}")

            toks = toks_cache[i].copy()
            toks[deleted_idx] = ""
            current_texts[i] = join_tokens(toks)

            toks_cache[i][deleted_idx] = ""
    
    current_pred = predict_torch(current_texts)

    flips = np.sum(np.argmax(current_pred, axis=1) != np.argmax(first_pred, axis=1))
    print(f"round={round_num:02d} flips={flips}/{N_TEXTS}")

PartitionExplainer explainer: 51it [00:53,  1.27s/it]                        


Altering unaltered samples
round=00 flips=0/50


PartitionExplainer explainer:  38%|███▊      | 19/50 [00:20<00:31,  1.01s/it]